# IR System — BM25 Retrieval
**Step 5:** BM25 retrieval for both datasets (k1=1.5, b=0.75).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/ir_system_data'
import os, sys

if not os.path.exists('/content/ir-system'):
    !git clone https://github.com/ghazal-mohammad/ir-system.git /content/ir-system
else:
    !cd /content/ir-system && git pull
sys.path.insert(0, '/content/ir-system')

!pip install ir-datasets==0.5.9 -q
print('ready')

In [ ]:
import json
from services.indexing_service import load_index, get_avg_doc_length
from services.bm25_service import retrieve_bm25, save_bm25_params
from services.preprocessing_service import preprocess
import ir_datasets

print('Loading data...')
index1 = load_index(f'{SAVE_DIR}/ct2021_index.pkl')
index2 = load_index(f'{SAVE_DIR}/msmarco_index.pkl')

with open(f'{SAVE_DIR}/ct2021_doc_lengths.json') as f:
    doc_lengths1 = json.load(f)
with open(f'{SAVE_DIR}/msmarco_doc_lengths.json') as f:
    doc_lengths2 = json.load(f)

avg_dl1 = get_avg_doc_length(doc_lengths1)
avg_dl2 = get_avg_doc_length(doc_lengths2)
print(f'CT2021 avg doc length: {avg_dl1:.1f}')
print(f'MSMARCO avg doc length: {avg_dl2:.1f}')

save_bm25_params(avg_dl1, f'{SAVE_DIR}/ct2021_bm25_params.pkl')
save_bm25_params(avg_dl2, f'{SAVE_DIR}/msmarco_bm25_params.pkl')
print('BM25 params saved')

In [ ]:
# Load queries
ds1 = ir_datasets.load('clinicaltrials/2021/trec-ct-2021')
queries1 = {q.query_id: q.text for q in ds1.queries_iter()}

ds2 = ir_datasets.load('msmarco-passage/trec-dl-2019')
queries2 = {q.query_id: q.text for q in ds2.queries_iter()}

print(f'CT2021: {len(queries1)} queries')
print(f'MSMARCO: {len(queries2)} queries')

In [ ]:
# Quick test on one query
sample_qid1 = list(queries1.keys())[0]
sample_query1 = queries1[sample_qid1]
tokens1 = preprocess(sample_query1)

res1 = retrieve_bm25(tokens1, index1, doc_lengths1, avg_dl1, top_k=10)
print(f'CT2021 query: "{sample_query1}"')
print('Top 10 BM25 results:')
for r in res1:
    print(f'  rank {r["rank"]}: {r["doc_id"]} (score={r["score"]})')

print()
sample_qid2 = list(queries2.keys())[0]
sample_query2 = queries2[sample_qid2]
tokens2 = preprocess(sample_query2)

res2 = retrieve_bm25(tokens2, index2, doc_lengths2, avg_dl2, top_k=10)
print(f'MSMARCO query: "{sample_query2}"')
print('Top 10 BM25 results:')
for r in res2:
    print(f'  rank {r["rank"]}: {r["doc_id"]} (score={r["score"]})')

In [ ]:
# Full retrieval for all queries
print('Running BM25 on all CT2021 queries...')
all_results1 = {}
for qid, qtext in queries1.items():
    tokens = preprocess(qtext)
    all_results1[qid] = retrieve_bm25(tokens, index1, doc_lengths1, avg_dl1, top_k=1000)
print(f'Done: {len(all_results1)} queries')

with open(f'{SAVE_DIR}/ct2021_bm25_results.json', 'w') as f:
    json.dump(all_results1, f)
print('CT2021 BM25 results saved')

print('Running BM25 on all MSMARCO queries...')
all_results2 = {}
for qid, qtext in queries2.items():
    tokens = preprocess(qtext)
    all_results2[qid] = retrieve_bm25(tokens, index2, doc_lengths2, avg_dl2, top_k=1000)
print(f'Done: {len(all_results2)} queries')

with open(f'{SAVE_DIR}/msmarco_bm25_results.json', 'w') as f:
    json.dump(all_results2, f)
print('MSMARCO BM25 results saved')

print('\n=== BM25 Complete ===')
print('Next: 06_retrieval_embedding.ipynb')